> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 11 · Notebook 03 — RAG I: chunking, point-in-time filtering and hybrid retrieval

**Sessions:** S3 (RAG I: ingestion, chunking & hybrid retrieval) · [Lesson plan](../../docs/lessons/PART_11_AI_NLP_COWORK.md) · graded labs in [`labs/part11/`](../../labs/part11/)

**You will:**
1. Chunk 10-K filings by section, and filter them to what was filed by a given date.
2. Score chunks with BM25, and fuse keyword and dense rankings with reciprocal rank fusion.
3. Measure retrieval with recall@k and MRR on a golden question set.
4. Find out why the right chunk is almost never ranked first, and fix it.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing calls the API: model outputs are recorded responses built as real `anthropic` Message objects, and the data is synthetic with known answers, so every check is exact and free.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p11lib.py is in notebooks/part11/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p11lib as p

p.use_course_style()

## 1. Chunks that know their date

`p.make_filings()` builds 15 synthetic 10-Ks (5 companies × 3 years, each filed in late February), an XBRL-like facts table and 18 golden questions. `p.chunk_filing` splits each filing at its `Item` headings and then into overlapping word windows; every chunk keeps its ticker, section and filing date. A backtest on 1 March 2024 may only read filings **filed on or before** that date: keep the ticker's chunks with `filed_at <= as_of`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pit_filter(chunks, ticker, as_of):
    as_of = pd.Timestamp(as_of)
    return [c for c in chunks if c.ticker == ticker and c.filed_at <= as_of]

filings, facts, questions = p.make_filings()
chunks = [c for f in filings for c in p.chunk_filing(f["text"], f["ticker"], f["form"], f["filed_at"])]
cases = [(chunks, "CRUX", "2024-03-01"), (chunks, "ACME", "2023-01-01"), (chunks, "BOLT", "2025-12-31")]
mine = [p.attempt(lambda *c: [x.chunk_id for x in pit_filter(*c)], *c) for c in cases]
mine = p.check("pit_filter", mine, [[x.chunk_id for x in p.pit_filter(*c)] for c in cases])
print(f"{len(filings)} filings → {len(chunks)} chunks;  CRUX chunks known on 2024-03-01: {len(mine[0])};  ACME on 2023-01-01: {len(mine[1])}")

## 2. BM25

Keyword search that rewards rare terms and saturates repeated ones. For each query term `q` found `f` times in a document of length `|d|`: add `idf(q) · f·(k1 + 1) / (f + k1·(1 − b + b·|d|/avgdl))` (`self.idf(q)`, `self.k1`, `self.b`, `self.avgdl` are given).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyBM25(p.BM25):
    def scores(self, query):
        out = np.zeros(self.N)
        for i, d in enumerate(self.docs):
            L = len(d)
            for q in query:
                f = d.count(q)
                if f:
                    out[i] += self.idf(q) * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * L / self.avgdl))
        return out

docs = [p.tokenize(c.text) for c in chunks]
queries = ["What was CRUX revenue in fiscal 2023?", "single supplier chips", "wind turbines utility customers"]
mine = [p.attempt(MyBM25(docs).scores, p.tokenize(q)) for q in queries]
mine = p.check("BM25.scores", mine, [p.BM25(docs).scores(p.tokenize(q)) for q in queries])
top = chunks[int(np.argmax(mine[1]))]
sentence = next(s for s in top.text.split(". ") if "supplier" in s)
print(f"best chunk for 'single supplier chips': {top.ticker} 10-K filed {top.filed_at:%Y-%m-%d}, {top.section} → …{sentence}…")

## 3. Hybrid retrieval

Dense embeddings (here an offline TF-IDF + SVD stand-in for sentence-transformers) find paraphrases; BM25 finds exact terms and numbers. **Reciprocal rank fusion** combines ranked lists without comparing their scores: each id gets `Σ 1/(k + rank)` over the lists (rank from 1, `k = 60`); sort by that score, descending (ties: first seen first, which `sorted` keeps).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

cases = [[["a", "b", "c"], ["c", "a", "d"]], [["x"], ["y"], ["y", "x"]], [["a", "b"], []]]
mine = [p.attempt(reciprocal_rank_fusion, c) for c in cases]
mine = p.check("reciprocal_rank_fusion", mine, [p.reciprocal_rank_fusion(c) for c in cases])
print(mine)

## 4. Evaluate on the golden set

`p.Retriever` applies the point-in-time filter **before** ranking and supports `bm25`, `dense` and `hybrid`. A chunk is relevant if it contains the question's answer phrase. **recall@k** = share of relevant chunks in the top k; **MRR** = mean of 1/(rank of the first relevant chunk).

In [ ]:
def relevant_ids(chunks, q):
    return {c.chunk_id for c in chunks if q["answer"] and q["answer"] in c.text and c.ticker == q["ticker"] and c.filed_at <= q["as_of"]}

def retrieval_table(chunks):
    retriever = p.Retriever(chunks)
    qs = [q for q in questions if q["answer"]]
    rel = [relevant_ids(chunks, q) for q in qs]
    rows = {}
    for mode in ("bm25", "dense", "hybrid"):
        got = [retriever.search(q["question"], q["ticker"], q["as_of"], 5, mode) for q in qs]
        rows[mode] = {"recall@1": p.recall_at_k(got, rel, 1), "recall@5": p.recall_at_k(got, rel, 5), "MRR": p.mrr(got, rel)}
    return pd.DataFrame(rows).T, retriever

table, retriever = retrieval_table(chunks)
display(table.round(3))
q = questions[0]
byid = {c.chunk_id: c for c in chunks}
print(f"Q: {q['question']}")
for rank, cid in enumerate(retriever.search(q["question"], q["ticker"], q["as_of"], 3), start=1):
    print(f"  {rank}. [{byid[cid].section}] {byid[cid].text[:80]}…")

Every answer is in the top 5, yet the right chunk is almost never first. The culprit is the one-line cover page ("ACME Annual Report on Form 10-K for fiscal 2023"): short, and it contains exactly the company and year every question asks about. Drop the `Preamble` chunks from the index and evaluate again:

In [ ]:
tuned, _ = retrieval_table([c for c in chunks if c.section != "Preamble"])
tuned.round(3)

Recall@1 and MRR jump. The fix came from **looking at what ranks first**, not from a better embedding model. On this corpus BM25 and hybrid beat the dense stand-in; measure on yours.

## Wrap-up

* Chunks carry metadata (ticker, section, filing date); filter point-in-time **before** ranking.
* Hybrid retrieval (BM25 + dense, fused with RRF) is a strong default.
* Evaluate with a golden set (recall@k, MRR) and read the top results.
* Graded version: `labs/part11/week39_nlp_rag` (`chunk_filing`, `pit_filter`, `BM25`, `reciprocal_rank_fusion`, `Retriever`) and Clinic W1 (the copilot evaluation).